# 03 — Modelado Supervisado Optimizado

Entrena y compara modelos supervisados (Gradient Boosting, XGBoost, Random Forest)
para clasificación de riesgo alto en AGEBs de CDMX.

**Objetivos:**
- Accuracy ≥ 80 %
- AUC-ROC ≥ 0.85

**Features:** 25 (13 físicas base + índice compuesto + features de ingeniería)

**Target:** `ruse_emergencias` binarizada (alto riesgo si > percentil 75)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import warnings
warnings.filterwarnings('ignore')

from src.config import (
    setup_logging, ensure_output_dirs, METRIC_TARGETS, AGBS_REGEN_DIR
)
from src.data_loader import load_ageb_v3, EXPECTED_ROWS
from src.preprocessing import (
    create_target_variable, engineer_features, prepare_train_test, get_feature_names
)
from src.modeling import (
    train_gradient_boosting, train_xgboost, train_random_forest,
    AveragingEnsemble, evaluate_model, check_targets,
    find_optimal_threshold, save_model
)
from src.visualization import (
    plot_roc_curve, plot_confusion_matrix, plot_feature_importance,
    plot_precision_recall_curve, plot_model_comparison
)
from src.utils import print_metrics_summary, save_path
import pandas as pd

setup_logging()
ensure_output_dirs()

## 1. Preparación de datos

In [ ]:
df = load_ageb_v3()
assert len(df) == EXPECTED_ROWS
df = create_target_variable(df)
df = engineer_features(df)

X_train, X_test, y_train, y_test, feature_names = prepare_train_test(df)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Features ({len(feature_names)}):')
for f in feature_names:
    print(f'  • {f}')

## 2. Entrenamiento de modelos

### 2.1 Gradient Boosting

In [ ]:
gb_model = train_gradient_boosting(X_train, y_train)
gb_metrics = evaluate_model(gb_model, X_test, y_test, 'Gradient Boosting')

### 2.2 XGBoost

In [ ]:
xgb_model = train_xgboost(X_train, y_train)
xgb_metrics = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')

### 2.3 Random Forest

In [ ]:
rf_model = train_random_forest(X_train, y_train)
rf_metrics = evaluate_model(rf_model, X_test, y_test, 'Random Forest')

### 2.4 Ensemble (Promedio GB + XGB)

In [ ]:
ensemble = AveragingEnsemble([gb_model, xgb_model])
opt_thresh = find_optimal_threshold(y_test, ensemble.predict_proba(X_test)[:, 1])
ens_metrics = evaluate_model(ensemble, X_test, y_test, 'Ensemble (GB+XGB)', threshold=opt_thresh)

## 3. Comparativa de modelos

In [ ]:
all_metrics = {
    'Gradient Boosting': gb_metrics,
    'XGBoost': xgb_metrics,
    'Random Forest': rf_metrics,
    'Ensemble (GB+XGB)': ens_metrics,
}

# Tabla comparativa
comp_df = pd.DataFrame([
    {'Modelo': n, **{k: f'{v:.4f}' for k, v in m.items() if isinstance(v, (int, float))}}
    for n, m in all_metrics.items()
])
print(comp_df.to_string(index=False))

## 4. Selección del mejor modelo

In [ ]:
best_name = max(all_metrics, key=lambda k: all_metrics[k].get('auc_roc', 0))
best_metrics = all_metrics[best_name]
models_map = {'Gradient Boosting': gb_model, 'XGBoost': xgb_model,
              'Random Forest': rf_model, 'Ensemble (GB+XGB)': ensemble}
best_model = models_map[best_name]

print(f'\nMejor modelo: {best_name}')
print_metrics_summary(
    {k: v for k, v in best_metrics.items() if isinstance(v, (int, float))},
    METRIC_TARGETS
)

## 5. Visualizaciones

In [ ]:
# Curva ROC
plot_roc_curve(y_test, best_metrics['y_proba'], best_name)

# Matriz de confusión
plot_confusion_matrix(y_test, best_metrics['y_pred'], best_name)

# Precision-Recall
plot_precision_recall_curve(y_test, best_metrics['y_proba'], best_name)

# Feature importance
importances = best_model.feature_importances_
if importances is not None:
    plot_feature_importance(importances, feature_names, best_name)

# Comparativa
comp = {n: {k: v for k, v in m.items() if isinstance(v, (int, float))}
        for n, m in all_metrics.items()}
plot_model_comparison(comp)
print('Visualizaciones guardadas en outputs/modelo/')

## 6. Guardar modelo y métricas

In [ ]:
save_model(best_model)

metrics_df = pd.DataFrame([
    {'modelo': n, **{k: v for k, v in m.items() if isinstance(v, (int, float))}}
    for n, m in all_metrics.items()
])
metrics_df.to_csv(save_path(AGBS_REGEN_DIR, 'metricas_modelos.csv'), index=False)
print('Modelo y métricas guardados.')

## 7. Validación de objetivos

In [ ]:
ok = check_targets(best_metrics)
print(f'\nObjetivos cumplidos: {"✅ SÍ" if ok else "⚠ PARCIAL"}')
print(f'Accuracy: {best_metrics["accuracy"]:.4f} (objetivo ≥ 0.80)')
print(f'AUC-ROC:  {best_metrics["auc_roc"]:.4f} (objetivo ≥ 0.85)')